# FSE — Train PhoBERT 3 lớp trên Google Colab + Kiểm định S_t

Notebook này là **Phase B + C** của bộ hướng dẫn *Colab → Tích hợp → Kiểm định → Báo cáo* cho repo
**`fse-financial-sentiment-engine`**. Toàn bộ ý nghĩa từng bước, cách đọc kết quả và Phase A/D/E
nằm trong file `FSE_HuongDan_Colab_TichHop_KiemDinh.docx`.

**Notebook làm 4 việc:**

1. Clone repo + cài môi trường — dùng **nguyên xi pipeline training của repo** (`src.training.train_fse_sft`), không viết lại code.
2. Train `VinAI/phobert-base-v2` phân loại 3 lớp (negative / neutral / positive) theo đúng `config/train_config.yaml` (3 seed, early stopping, class weights tự tính).
3. **Kiểm định 4 lớp**: K1 test set + baseline đa số · K2 gold người (tùy chọn) · K3 S_t model vs S_t Gemini · K4 8 cặp tương quan + hiệu chỉnh BH-FDR + lag.
4. Lưu toàn bộ artifact vào Google Drive để mang về máy tích hợp (Phase D).

**Trước khi chạy (Phase A đã xong trên máy local):**

- Đã upload `texts_with_dates.csv` lên Drive trong thư mục `fse_colab/`
- Runtime → Change runtime type → **T4 GPU**

| Chế độ | SEEDS | Thời gian ước tính (T4) |
|---|---|---|
| Thử nhanh lần đầu | `"42"` | ~35–45 phút |
| Lấy số liệu báo cáo | `"42,7,2026"` | ~1h15–1h45 |

> Nếu runtime bị ngắt giữa chừng: chạy lại từ CELL 2 (mount Drive) + CELL 3 (clone + cài), sau đó chạy thẳng CELL 6 — cache tách từ còn lưu trên đĩa nếu bạn đã backup, còn không underthesea sẽ tách lại (~5–10 phút).

In [ ]:
# CELL 1 — Kiểm tra GPU
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Chưa có GPU! Runtime → Change runtime type → T4 GPU, rồi chạy lại cell này."
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# CELL 2 — Mount Drive + CẤU HÌNH (sửa SEEDS trước khi chạy)
from google.colab import drive
drive.mount('/content/drive')

# ============ CẤU HÌNH ============
SEEDS = "42,7,2026"                              # thử nhanh: "42"  |  báo cáo: "42,7,2026"
DRIVE_DIR = "/content/drive/MyDrive/fse_colab"   # thư mục làm việc trên Drive
DATA_CSV = f"{DRIVE_DIR}/texts_with_dates.csv"   # file xuất từ Phase A
# ==================================

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Đã mount. Thư mục làm việc:", DRIVE_DIR)
print("SEEDS =", SEEDS)

In [ ]:
# CELL 3 — Clone repo + cài phụ thuộc còn thiếu
REPO_URL = "https://github.com/phmldtngyn/fse-financial-sentiment-engine.git"
!git clone {REPO_URL} /content/fse
%cd /content/fse

# Colab có sẵn torch / transformers / pandas / scikit-learn.
# KHÔNG cài requirements.txt của repo (đã pin version, dễ xung đột Colab) — chỉ cài thiếu:
!pip -q install underthesea pyarrow

import torch, transformers, pandas, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
!git log --oneline -1

In [ ]:
# CELL 4 — Dữ liệu train/val/test có sẵn trong repo (đã chia theo thời gian 80/10/10)
import pandas as pd, sys
sys.path.insert(0, ".")

from src.data.dataset import verify_splits  # module kiểm tra split của chính repo

dfs = {}
for name in ("train", "val", "test"):
    dfs[name] = pd.read_parquet(f"data/processed/{name}_3class.parquet")
    dist = dfs[name]["label"].value_counts().sort_index().to_dict()
    print(f"{name:5s}: {len(dfs[name]):5d} mẫu | phân bố lớp {dist}")

print("\nTổng cộng:", sum(len(d) for d in dfs.values()), "mẫu (kỳ vọng ~6.458)")
print("\nKiểm tra split bằng verify_splits() của repo (parquet không còn cột date")
print("nên phần kiểm tra thời gian sẽ báo cảnh báo — đây là hành vi đúng):")
for w in verify_splits(dfs["train"], dfs["val"], dfs["test"], date_col=None):
    print("  -", w)

In [ ]:
# CELL 5 — Nạp dữ liệu Phase A (text + date) từ Drive + đối chiếu với parquet train
import pandas as pd, re

df_all = pd.read_csv(DATA_CSV)
assert {"text", "date"}.issubset(df_all.columns), \
    f"CSV phải có cột text + date. Hiện có: {list(df_all.columns)}"
df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
df_all = df_all.dropna(subset=["date", "text"]).reset_index(drop=True)
df_all["text"] = df_all["text"].astype(str)
print(f"texts_with_dates: {len(df_all):,} dòng | "
      f"{df_all['date'].min():%Y-%m-%d} → {df_all['date'].max():%Y-%m-%d}")

# Đối chiếu text CSV ↔ parquet train (chuẩn hóa giống _norm_text của repo)
def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", str(t).lower())).strip()

train_set = {norm(t) for t in dfs["train"]["text"]}
csv_set = {norm(t) for t in df_all["text"]}
match = len(train_set & csv_set) / max(1, len(train_set))
print(f"Khớp text CSV ↔ train parquet: {match:.1%} (kỳ vọng ≥ 90%)")
assert match >= 0.90, "CSV không khớp dữ liệu train — hãy làm lại Phase A (xem Phụ lục guide)." 

## Train (multi-seed, đúng pipeline repo)

CELL 6 gọi **nguyên xi** `src.training.train_fse_sft` của repo với `config/train_config.yaml`:

- **3 seed** `42, 7, 2026` — mỗi seed một child run, chọn model tốt nhất theo **val F1-macro**, báo cáo mean ± std.
- **Early stopping** patience 3 (val F1 không tăng 3 epoch liên tiếp thì dừng sớm).
- **Class weights auto** — nghịch đảo tần suất từ tập train. Với phân bố hiện tại (626 / 3423 / 1117), log sẽ in ra xấp xỉ `[2.75, 0.50, 1.54]`.
- **AMP fp16** tự bật trên GPU; dynamic padding theo batch (nhanh hơn padding cứng 256).
- Checkpoint lưu kèm metadata: seed, data hash, config snapshot — lần nào cũng biết model sinh ra từ đâu.

Lần đầu chạy sẽ tải `VinAI/phobert-base-v2` (~720 MB) và tách từ underthesea cho ~6.4k câu (~5–10 phút, có cache đĩa — các seed sau và lần chạy sau dùng lại cache).

In [ ]:
# CELL 6 — TRAIN (pipeline chuẩn của repo). Đừng đóng tab trong lúc chạy!
#   Mỗi seed ~20–30 phút trên T4 (có thể dừng sớm nhờ early stopping).
!python -m src.training.train_fse_sft --config config/train_config.yaml --seeds {SEEDS} --no-mlflow --tag colab

In [ ]:
# CELL 7 — Xem kết quả: tổng hợp multi-seed + báo cáo test set
import json, pathlib
ck = pathlib.Path("checkpoints/fse_sft")

summary = json.loads((ck / "training_summary.json").read_text(encoding="utf-8"))
print("Val F1-macro theo seed :", summary["val_f1_by_seed"])
print(f"Mean ± std            : {summary['val_f1_mean']:.4f} ± {summary['val_f1_std']:.4f}")
print("Best seed             :", summary["best_seed"])
print(f"TEST: acc={summary['test']['accuracy']:.4f} | f1_macro={summary['test']['f1_macro']:.4f} "
      f"| f1_weighted={summary['test']['f1_weighted']:.4f} | n={summary['test']['n_samples']}")

print("\n" + "=" * 72 + "\nBÁO CÁO TEST SET CHI TIẾT (model được chọn)\n" + "=" * 72)
print((ck / "selected" / "eval_report.md").read_text(encoding="utf-8"))

## K1 — Đánh giá chi tiết test set + baseline đa số

Model lại được nạp từ checkpoint qua `load_checkpoint` của repo (cùng đường với API serving) và chạy
dự đoán trên test set. So sánh với **baseline đa số** (luôn đoán lớp neutral — lớp chiếm ~65%):
một model có giá trị phải thắng baseline này rõ rệt. Đây chính là mảnh "no-signal baseline" mà phần
tương quan cũ của repo còn thiếu.

In [ ]:
# CELL 8 — K1: dự đoán test set + baseline đa số + confusion matrix
import torch, numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from src.training.evaluate import load_checkpoint
from src.data.dataset import ensure_segmented, Collator

device = torch.device("cuda")
model, meta = load_checkpoint("checkpoints/fse_sft/selected/best.pt", device)
tok = AutoTokenizer.from_pretrained(meta.get("backbone", "VinAI/phobert-base-v2"))


def predict_texts(texts, bs=64, desc="predict"):
    """Text thô → tự tách từ (auto, có cache) → tokenize → softmax 3 lớp."""
    seg = ensure_segmented([str(t) for t in texts], mode="auto",
                           cache_dir="data/cache/segmentation")
    enc = tok(seg, truncation=True, max_length=256, padding=False)
    items = [{"input_ids": ids, "attention_mask": am, "labels": 0}
             for ids, am in zip(enc["input_ids"], enc["attention_mask"])]
    loader = DataLoader(items, batch_size=bs, shuffle=False, collate_fn=Collator(tok))
    probs = []
    model.eval()
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for batch in tqdm(loader, desc=desc):
            out = model(input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device))
            logits = out.logits if hasattr(out, "logits") else out
            probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy()


NAMES = ["negative", "neutral", "positive"]
probs_test = predict_texts(dfs["test"]["text"], desc="test set")
y_true = dfs["test"]["label"].to_numpy()
y_pred = probs_test.argmax(1)

print(classification_report(y_true, y_pred, target_names=NAMES, digits=4, zero_division=0))

majority = np.bincount(y_true).argmax()
acc_model = float((y_pred == y_true).mean())
acc_major = float((y_true == majority).mean())
print(f"Accuracy model          : {acc_model:.4f}")
print(f"Baseline đa số ({NAMES[majority]:>8s}): {acc_major:.4f}  → model hơn baseline {acc_model - acc_major:+.4f}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2), constrained_layout=True)
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), NAMES, rotation=20); ax.set_yticks(range(3), NAMES)
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Nhãn thật")
ax.set_title(f"Confusion matrix — test set (n={len(y_true)})")
fig.savefig("confusion_matrix_test.png", dpi=150)
plt.show()
print("Đã lưu confusion_matrix_test.png")

## K2 — Đánh giá trên nhãn người (TÙY CHỌN)

Repo đã có 2.000 câu mẫu vàng (`data/gold/all_sampled.csv`) nhưng **chưa có nhãn người** đi kèm.
Nếu bạn đã chấm người xong (file `gold_labeled.csv` với cột `text` + `gold_label` ∈ {0,1,2}, đặt ở
`Drive/fse_colab/`), cell dưới đo F1 trên nhãn người — **đây mới là con số trung thực nhất** vì nhãn
Gemini chỉ là teacher, không phải ground truth. Chưa chấm xong thì bỏ qua cell này không sao.

In [ ]:
# CELL 9 — K2 (tùy chọn): so model với nhãn người nếu đã có gold_labeled.csv
import os

GOLD_CSV = f"{DRIVE_DIR}/gold_labeled.csv"
if os.path.exists(GOLD_CSV):
    gold = pd.read_csv(GOLD_CSV)
    gold = gold.dropna(subset=["text", "gold_label"])
    gold["gold_label"] = gold["gold_label"].astype(int)
    probs_gold = predict_texts(gold["text"], desc="gold")
    y_gold = gold["gold_label"].to_numpy()
    y_hat = probs_gold.argmax(1)
    print(f"Gold set: {len(gold)} câu | phân bố {np.bincount(y_gold, minlength=3).tolist()}")
    print(classification_report(y_gold, y_hat, target_names=NAMES, digits=4, zero_division=0))
else:
    print("Chưa có gold_labeled.csv → BỎ QUA K2 (bình thường nếu chưa chấm người xong).")

## K3 — S_t của riêng bạn vs S_t Gemini

Đây là phần trả lời câu hỏi *"train bộ param của riêng mình rồi đo xem giá trị trả ra đúng bao nhiêu %"*:

- Dự đoán **toàn bộ ~6.458 tin** (có ngày) bằng model vừa train → argmax → map về pseudo-score
  `{0: -0.7, 1: 0.0, 2: +0.7}` (đúng quy ước `api/routes/sentiment.py`) → **trung bình theo ngày** = `S_t_model`.
- So với `S_t` Gemini hiện có trong `data/processed/banking_with_sentiment.parquet` (2.492 ngày).
- Lưu ý trung thực: phần tin thuộc train set là **in-sample** với model — so sánh này đo **tính nhất quán
  pipeline**, còn khả năng tổng quát hóa đã được đo độc lập ở K1 (test set).

In [ ]:
# CELL 10 — K3: S_t_model theo ngày vs S_t Gemini + Pearson/Spearman + biểu đồ
from scipy import stats

CLASS_TO_SCORE = {0: -0.7, 1: 0.0, 2: +0.7}

probs_all = predict_texts(df_all["text"], bs=64, desc="S_t (toàn bộ tin)")
df_pred = pd.DataFrame({
    "date": df_all["date"].dt.strftime("%Y-%m-%d").values,
    "pred": probs_all.argmax(1),
})
df_pred["score"] = df_pred["pred"].map(CLASS_TO_SCORE)

# S_t_model = trung bình các tin trong ngày (cùng công thức AVG(Sentiment_NLP) của repo)
st_model = df_pred.groupby("date")["score"].agg(["mean", "count"]).reset_index()
st_model.columns = ["trade_date", "S_t_model", "n_model"]

st_gem = pd.read_parquet("data/processed/banking_with_sentiment.parquet")
m = st_gem.merge(st_model, on="trade_date", how="inner").sort_values("trade_date").reset_index(drop=True)
print(f"S_t Gemini: {len(st_gem)} ngày | S_t model: {len(st_model)} ngày | khớp theo ngày: {len(m)}")

r_p, p_p = stats.pearsonr(m["S_t"], m["S_t_model"])
r_s, p_s = stats.spearmanr(m["S_t"], m["S_t_model"])
print(f"Pearson   r = {r_p:+.4f}  (p = {p_p:.2e})")
print(f"Spearman  ρ = {r_s:+.4f}  (p = {p_s:.2e})")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
axes[0].plot(m["S_t"].values, lw=0.8, label="S_t Gemini")
axes[0].plot(m["S_t_model"].values, lw=0.8, alpha=0.8, label="S_t Model")
axes[0].set_title("S_t theo ngày"); axes[0].legend(fontsize=8)
axes[1].scatter(m["S_t"], m["S_t_model"], s=6, alpha=0.5)
axes[1].set_xlabel("S_t Gemini"); axes[1].set_ylabel("S_t Model")
axes[1].set_title(f"Scatter — Pearson r = {r_p:+.3f}")
roll = m["S_t"].rolling(60).corr(m["S_t_model"])
axes[2].plot(roll.values, lw=0.9, color="teal")
axes[2].axhline(0, ls="--", lw=0.6, color="gray"); axes[2].set_ylim(-1, 1)
axes[2].set_title("Tương quan trượt 60 ngày")
fig.savefig("st_model_vs_gemini.png", dpi=150)
plt.show()

m.to_parquet("banking_with_sentiment_MODEL.parquet", index=False)
st_model.to_csv("st_model_daily.csv", index=False)
print("Đã lưu banking_with_sentiment_MODEL.parquet + st_model_daily.csv")

## K4 — Tương quan thị trường: Gemini vs Model, có hiệu chỉnh FDR

Chạy lại **đúng 8 cặp** của `scripts/correlation_analysis.py` nhưng trên cả 2 phiên bản S_t, rồi
hiệu chỉnh Benjamini–Hochberg bằng **chính module `src/analysis/correlation_stats.py` của repo**.
Kèm lag 0–2 (S_t đi trước return — đúng nghĩa dự báo) và độ ổn định out-of-sample 70/30.

**Cách đọc**: dấu và độ lớn các hệ số phải **giữ nguyên bậc thang** giữa 2 phiên bản. Nếu model cùng
hướng với Gemini nhưng hệ số nhích nhẹ → pipeline model không phá kết luận của báo cáo; nếu lật dấu
hoặc sụp độ lớn → phải soi lại trước khi đưa vào báo cáo.

In [ ]:
# CELL 11 — K4: 8 cặp × 2 phiên bản S_t + BH-FDR + lag + OOS (module của repo)
from src.analysis.correlation_stats import benjamini_hochberg, lag_correlations, out_of_sample_corr

def pairs_table(df, col_s):
    feats = [("total_volume", "S_t vs Volume"), ("log_volume", "S_t vs log(Volume)"),
             ("avg_return", "S_t vs Return"), ("avg_range", "S_t vs Intraday Range"),
             ("abs_return", "S_t vs |Return|"), ("avg_range", "|S_t| vs Intraday Range"),
             ("log_volume", "|S_t| vs log(Volume)"), ("abs_return", "|S_t| vs |Return|")]
    s_abs = df[col_s].abs()
    rows = []
    for feat, label in feats:
        x = s_abs if label.startswith("|S_t|") else df[col_s]
        r, p = stats.pearsonr(x, df[feat])
        rs, _ = stats.spearmanr(x, df[feat])
        rows.append({"feature": label, "pearson_r": round(float(r), 4), "p": float(p),
                     "spearman_r": round(float(rs), 4)})
    return pd.DataFrame(rows)

t_gem = pairs_table(m, "S_t")
t_mod = pairs_table(m, "S_t_model")
q = benjamini_hochberg(list(t_gem["p"]) + list(t_mod["p"]))
t_gem["q_fdr"] = [round(v, 5) for v in q[:len(t_gem)]]
t_mod["q_fdr"] = [round(v, 5) for v in q[len(t_gem):]]

cmp = t_gem.merge(t_mod, on="feature", suffixes=("_gemini", "_model"))
cmp["diff_r"] = (cmp["pearson_r_model"] - cmp["pearson_r_gemini"]).round(4)
print(cmp[["feature", "pearson_r_gemini", "pearson_r_model", "diff_r",
           "q_fdr_gemini", "q_fdr_model"]].to_string(index=False))
cmp.to_csv("correlation_gemini_vs_model.csv", index=False)

lags_gem = lag_correlations(m["S_t"].to_numpy(), m["avg_return"].to_numpy(), lags=(0, 1, 2))
lags_mod = lag_correlations(m["S_t_model"].to_numpy(), m["avg_return"].to_numpy(), lags=(0, 1, 2))
print("\nLag S_t → Return (số dương = sentiment đi trước return):")
for lag in (0, 1, 2):
    print(f"  lag {lag}: Gemini r={lags_gem[lag]['corr']}  |  Model r={lags_mod[lag]['corr']}")

oos = out_of_sample_corr(m["S_t_model"].to_numpy(), m["avg_return"].to_numpy())
print("\nOOS 70/30 — S_t_model vs Return:", oos)
print("\nĐã lưu correlation_gemini_vs_model.csv")

In [ ]:
# CELL 12 — Lưu toàn bộ artifact vào Drive (Phase D sẽ dùng các file này)
import shutil, json

save_dir = f"{DRIVE_DIR}/results"
os.makedirs(save_dir, exist_ok=True)

pairs = [
    ("checkpoints/fse_sft/selected/best.pt", "best.pt"),
    ("checkpoints/fse_sft/training_summary.json", "training_summary.json"),
    ("checkpoints/fse_sft/selected/eval_report.json", "eval_report.json"),
    ("checkpoints/fse_sft/selected/eval_report.md", "eval_report.md"),
    ("banking_with_sentiment_MODEL.parquet", "banking_with_sentiment_MODEL.parquet"),
    ("st_model_daily.csv", "st_model_daily.csv"),
    ("correlation_gemini_vs_model.csv", "correlation_gemini_vs_model.csv"),
    ("confusion_matrix_test.png", "confusion_matrix_test.png"),
    ("st_model_vs_gemini.png", "st_model_vs_gemini.png"),
]
for src, dst in pairs:
    shutil.copy2(src, f"{save_dir}/{dst}")

# metrics từng epoch của seed được chọn + dự đoán chi tiết test set
best_seed = json.loads((ck / "training_summary.json").read_text(encoding="utf-8"))["best_seed"]
shutil.copy2(f"checkpoints/fse_sft/seed{best_seed}/metrics.jsonl", f"{save_dir}/metrics_best_seed.jsonl")
pd.DataFrame({
    "y_true": y_true, "y_pred": y_pred,
    "p_neg": probs_test[:, 0], "p_neu": probs_test[:, 1], "p_pos": probs_test[:, 2],
    "text": dfs["test"]["text"].values,
}).to_csv(f"{save_dir}/predictions_test.csv", index=False)

# Cache tách từ — lần train sau khỏi chờ underthesea
shutil.make_archive(f"{save_dir}/segmentation_cache", "zip", "data/cache/segmentation")

print("ĐÃ LƯU XONG →", save_dir)
!ls -lh "{save_dir}" 

## Xong Colab — Mang gì về máy (Phase D)

Trong `Drive/fse_colab/results/` có 10 file. **Bắt buộc tải về máy:**

| File | Đặt vào đâu (máy local) |
|---|---|
| `best.pt` | `checkpoints/fse_sft/selected/best.pt` trong repo local |
| `training_summary.json`, `eval_report.json/.md`, `metrics_best_seed.jsonl` | giữ lại làm bằng chứng cho báo cáo |
| `banking_with_sentiment_MODEL.parquet`, `st_model_daily.csv`, `correlation_gemini_vs_model.csv` | `data/processed/` |
| `confusion_matrix_test.png`, `st_model_vs_gemini.png` | chèn vào báo cáo |

**10 con số cần ghi lại cho Phase E** (đều có trong `training_summary.json` + output các cell):
val F1 từng seed, mean ± std, best seed, test acc, test F1-macro, test F1-weighted, Pearson r(S_t),
Spearman ρ(S_t), diff_r lớn nhất trong 8 cặp, số ngày khớp S_t.

Tiếp theo: mở `FSE_HuongDan_Colab_TichHop_KiemDinh.docx` → **Phase D (tích hợp máy local)**.